In [1]:
import os

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

# omegaconf é usado pelo EasyTPP para carregar configurações YAML
!pip install omegaconf -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, sys, math, random, hashlib, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from scipy import stats

TRAIN_LEN = 50      
EXTRAP_FACTORS = [2, 5, 10] 
N_SEEDS = 5       
EPOCHS = 500     
PATIENCE = 30      
BASE_SEED = 42      
NUM_TYPES = 2       
PAD_ID = NUM_TYPES 

PROC_SLOW = dict(
    mu    = np.array([0.3, 0.3]),
    alpha = np.array([[0.008, 0.006], [0.006, 0.008]]),
    beta  = 0.02,
    label = 'Decaimento LENTO',
)

PROC_FAST = dict(
    mu    = np.array([0.4, 0.4]),
    alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
    beta  = 0.50,
    label = 'Decaimento RÁPIDO',
)

PROCESSES = [('slow', PROC_SLOW), ('fast', PROC_FAST)]

# Cores dos modelos nos gráficos
COR_ROTHP = '#4C72B0'   # azul para RoTHP
COR_HOTHP = '#C44E52'   # vermelho para HoTHP

# ── Funções auxiliares de reprodutibilidade ────────────────────────────────
def set_seed(seed):
    """Fixa todas as sementes aleatórias para garantir reprodutibilidade."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    """Gera uma semente diferente para cada combinação de argumentos.
    Evita usar a mesma semente para treino de modelos diferentes."""
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)

# ── Configuração do dispositivo (GPU ou CPU) ───────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# AMP = Automatic Mixed Precision: acelera o treino na GPU usando float16
USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer
import easy_tpp.model.torch_model.torch_rothp as rothp_module

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

In [3]:
def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(100):  # tenta até 100 vezes para garantir min_ev eventos
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9:
                break
            
            t += rng.exponential(1.0 / lam_bar) # proximo tempo candidato
            if t >= horizon:
                break

            # Recalcula a taxa real no tempo candidato
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
                        
            if rng.uniform() <= cand.sum() / lam_bar: 
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    """
    Converte uma lista de sequências de eventos em tensores PyTorch.
    Aplica normalização por sequência: divide os tempos pelo gap médio,
    para que o intervalo médio entre eventos seja ~1 (mesma escala para todos).
    O gap é a média dos deltas.. assim, os tempos ficam próximos, independente da escala original de tempo
    """
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])         # garante ordem temporal
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)  # tempos absolutos
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)     # tipos
        d = torch.zeros_like(t)                        # gaps (diferenças entre tempos)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)              # gap médio (evita divisão por zero)
        t = (t - t[0]) / mg                            # normaliza tempos
        d = d / mg                                     # normaliza gaps
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch):
    """
    Função de collation: agrupa sequências de tamanhos variados em batches.
    Adiciona padding para igualar o comprimento de todas as sequências do batch.
    Cria a máscara causal (cada evento só pode ver eventos anteriores).
    """
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)  # comprimento da maior sequência no batch

    # Inicializa tensores com zeros ou padding
    t_pad  = torch.zeros(B, L)
    d_pad  = torch.zeros(B, L)
    k_pad  = torch.full((B, L), PAD_ID, dtype=torch.long) # PAD_ID é o len(k) + 1, que é um numero impossível
    npm    = torch.zeros(B, L)   # máscara de posições reais (não-padding mask)

    # Máscara causal: impede que o modelo olhe para frente no tempo
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)

    for i, item in enumerate(batch):
        sl = len(item['time_seqs']) # sequence length.. tamanho da sequencia
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl]   = 1.0  # marca posições reais (non-padding mask)
        m = causal.clone()
        m[:, sl:] = True     # mascara posições de padding nas colunas
        m[sl:, :] = True # nas linhas
        attn[i] = m

    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    """Cria um DataLoader PyTorch a partir de uma lista de sequências."""
    g = None
    if shuffle and seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


rng = np.random.default_rng(run_seed('main', 'data'))
datasets = {}

for pname, proc in PROCESSES: # process names... slow e fast
    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']

    raw = {
        'train': [simulate_hawkes(rng, mu, alpha, beta, 50.0, 10, TRAIN_LEN) for _ in range(500)],
        'val':   [simulate_hawkes(rng, mu, alpha, beta, 50.0, 10, TRAIN_LEN) for _ in range(150)],
        'short': [simulate_hawkes(rng, mu, alpha, beta, 50.0, 10, TRAIN_LEN) for _ in range(200)],
    }

    # Dados de TESTE OOD: sequências muito mais longas do que o treino
    for f in EXTRAP_FACTORS:
        raw[f'extrap_{f}x'] = [
            simulate_hawkes(rng, mu, alpha, beta,
                            50.0 * f, # horizonte de tempo maior
                            TRAIN_LEN + 5, # exige ao menos TRAIN_LEN+5 eventos
                            TRAIN_LEN * f) # e limita a TRAIN_LEN*f eventos
            for _ in range(200)
        ]

    datasets[pname] = {k: to_tensors(v) for k, v in raw.items()}

    # Ccalcular o β_normalizado real (beta * gap_médio) para informar nos gráficos..  βnorm = β×Δtˉ
    # Como normalizamos os dados, precisamos noramlizar beta. Coerente? (Cesar)
    # mu e alpha não precisam de normalização pois são intensidades e não taxas, não entram em nenhum expoente
    gaps = []
    for seq in raw['train']:
        ts = sorted([t for t, _ in seq])
        gaps.extend([ts[i] - ts[i-1] for i in range(1, len(ts))])
    mg  = np.mean(gaps)
    bn  = beta * mg
    infl = math.exp(-bn * (TRAIN_LEN * max(EXTRAP_FACTORS) - 1))
    print(f'  {pname:4s}  beta={beta:.2f}  gap_médio={mg:.3f}  β_norm={bn:.4f}')

  slow  beta=0.02  gap_médio=1.245  β_norm=0.0249  influência@10x=0.0000
  fast  beta=0.50  gap_médio=1.011  β_norm=0.5056  influência@10x=0.0000


In [4]:
config = ModelConfig(**{
    'hidden_size':   32,  
    'num_layers':    2,   
    'num_heads':     2,   
    'dropout_rate':  0.1,
    'num_event_types': NUM_TYPES,
    'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID,
    'time_emb_size': 32,
    'use_ln': True, # layer normalization (lembrar)
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'Simplificado',
    # campos nececssários da bilbioteca, mas nao sao usados 
    'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                 'patience_counter': 5, 'num_samples_boundary': 5,
                 'dtime_max': 5.0, 'num_step_gen': 1},
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})

def eval_nll(model, dl):
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast(): # _autocast() é pra precisão mista (float16/bfloat16 na GPU quando disponível)
                l, n = model.loglike_loss(batch) # NLL somada do batch e n é o numero de eventos do batch (sem padding)
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9) # divide (com segurança) NLL pelo numero de eventos 


def eval_ood_nll(model, dl):
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            # Zera a máscara nas posições IN-DISTRIBUTION (mantém só OOD)
            mask = npm.clone()
            mask[:, :TRAIN_LEN] = 0.0 # zera todas as linhas e todas as colunas até chegar noo train_len
            if mask.sum() == 0:
                continue  # batch sem eventos OOD, pula
            with _autocast():
                l, n = model.loglike_loss([t, d, k, mask, attn])
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def eval_ood_nll_full(model, dl):
    """NLL sobre a sequência OOD inteira (in-dist + OOD), sem mascarar eventos iniciais.
    Forma mais comum na literatura: avalia o modelo em toda a sequência longa.
    Comparar com eval_ood_nll permite checar se excluir os eventos in-dist faz diferença."""
    return eval_nll(model, dl)


def train_model(cls, train_dl, val_dl, lr, base_seed):
    set_seed(base_seed)
    m = cls(config).to(device)

    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4) # AdamW?

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5) # reduzir o learning rate quando para de melhorar

    # Scaler para mixed precision (só na GPU) float16 na maioria dos calculos e float32 apenas onde precisão é crítica
    scaler = _Scaler(enabled=True) if USE_AMP else None

    # variáveis do early stopping
    best_val  = float('inf')
    best_state = None
    no_imp = 0  # contador de épocas sem melhora

    for ep in range(EPOCHS):
        # treino
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = m.loglike_loss(batch)
                nll  = l / (n + 1e-9)  # NLL média por evento
            if not torch.isnan(nll): # se a função hiperbolica se tornar instável, ignorar todo o passo
                if scaler:
                    scaler.scale(nll).backward() # escala a NLL pra evitar qeu gradientes virem 0 em float16
                    scaler.unscale_(opt) # desfaz a escala antes do clipping
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)  # evita explosão do gradiente
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()

        # validação
        v = eval_nll(m, val_dl)
        sched.step(v)

        if v < best_val - 1e-4: # melhora significativa
            best_val   = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    # Restaura o melhor modelo visto durante o treino
    m.load_state_dict(best_state)
    return m, best_val


In [5]:
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
results = []  # lista que vai acumular todos os resultados

for pname, proc in PROCESSES:
    print(f'Processo: {proc["label"]}')

    # DataLoaders de avaliação (criado apenas 1 vez)
    val_dl   = make_loader(datasets[pname]['val'],   64)
    short_dl = make_loader(datasets[pname]['short'], 64)
    ext_dls  = {f: make_loader(datasets[pname][f'extrap_{f}x'], 16)
                for f in EXTRAP_FACTORS}

    for seed_idx, seed in enumerate(seeds):
        print(f'  Seed {seed_idx+1}/{N_SEEDS}', end='  ')

        # DataLoader de treino com seed especifica
        train_dl = make_loader(datasets[pname]['train'], 64,
                               shuffle=True, seed=run_seed(pname, seed))

        # RoTHP
        rothp, rv = train_model(
            RoTHP, train_dl, val_dl,
            lr=1e-3,
            base_seed=run_seed(pname, 'rothp', seed)
        )

        # HoTHP 
        hothp, hv = train_model(
            HoTHP, train_dl, val_dl,
            lr=5e-4, # testes com lr menor.. (Cesar?)
            base_seed=run_seed(pname, 'hothp', seed)
        )

        # sequencias do mesmo comprimento do treino
        r_short = eval_nll(rothp, short_dl)
        h_short = eval_nll(hothp, short_dl)
        print(f'val NLL: RoTHP={rv:.4f}  HoTHP={hv:.4f}')

        # NLL OOD para cada fator de extrapolação
        for f in EXTRAP_FACTORS:
            r_ood      = eval_ood_nll(rothp, ext_dls[f])       # NLL RoTHP — só eventos OOD
            h_ood      = eval_ood_nll(hothp, ext_dls[f])       # NLL HoTHP — só eventos OOD
            r_ood_full = eval_ood_nll_full(rothp, ext_dls[f])  # NLL RoTHP — sequência inteira
            h_ood_full = eval_ood_nll_full(hothp, ext_dls[f])  # NLL HoTHP — sequência inteira
            results.append(dict(
                proc       = pname,
                seed       = seed,
                factor     = f,
                r_short    = r_short,    # NLL RoTHP na distribuição de treino
                h_short    = h_short,    # NLL HoTHP na distribuição de treino
                r_ood      = r_ood,      # NLL RoTHP — só eventos OOD (mascarado)
                h_ood      = h_ood,      # NLL HoTHP — só eventos OOD (mascarado)
                r_ood_full = r_ood_full, # NLL RoTHP — sequência OOD inteira
                h_ood_full = h_ood_full, # NLL HoTHP — sequência OOD inteira
                r_deg      = r_ood - r_short,      # degradação RoTHP (mascarada)
                h_deg      = h_ood - h_short,      # degradação HoTHP (mascarada)
                r_deg_full = r_ood_full - r_short, # degradação RoTHP (sequência inteira)
                h_deg_full = h_ood_full - h_short, # degradação HoTHP (sequência inteira)
                r_val      = rv,  # NLL de validação do RoTHP
                h_val      = hv,  # NLL de validação do HoTHP
            ))

df = pd.DataFrame(results)

Processo: Decaimento LENTO
  Seed 1/5  val NLL: RoTHP=1.6808  HoTHP=1.6842
  Seed 2/5  val NLL: RoTHP=1.6804  HoTHP=1.7149
  Seed 3/5  val NLL: RoTHP=1.6806  HoTHP=1.6938
  Seed 4/5  val NLL: RoTHP=1.6815  HoTHP=1.6851
  Seed 5/5  

KeyboardInterrupt: 

In [ ]:
x = np.arange(len(EXTRAP_FACTORS))
factor_labels = [f'{f}×' for f in EXTRAP_FACTORS] # (2x, 5x, 10x)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
for col, (pname, proc) in enumerate(PROCESSES):
    sub = df[df['proc'] == pname] 
    ax  = axes[col]

    for col_key, label, color in [
        ('r_deg', 'RoTHP', COR_ROTHP),
        ('h_deg', 'HoTHP', COR_HOTHP),
    ]:
        grp = sub.groupby('factor')[col_key]
        m   = grp.mean().reindex(EXTRAP_FACTORS)  # média sobre sementes
        s   = grp.std().reindex(EXTRAP_FACTORS)   # desvio padrão sobre sementes

        ax.plot(x, m.values, 'o-', color=color, lw=2.5, ms=8, label=label)
        ax.fill_between(x, m - s, m + s, color=color, alpha=0.15)

    ax.set_xticks(x)
    ax.set_xticklabels(factor_labels, fontsize=12)
    ax.set_xlabel('Fator de extrapolação', fontsize=11)
    ax.set_ylabel('ΔNLL  (NLL_OOD − NLL_inDist)', fontsize=11)
    ax.set_title(proc['label'], fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f'HoTHP vs RoTHP — Degradação com Extrapolação\n'
    f'Treino: ≤{TRAIN_LEN} eventos  →  Teste: até {TRAIN_LEN * max(EXTRAP_FACTORS)} eventos  |  n={N_SEEDS} sementes',
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# NLL de validação por semente
# Cada ponto é uma seed. Permite ver se algum modelo teve seeds que não convergiram.
# r_val e h_val são iguais para todas as linhas de uma mesma seed — pegamos uma por (proc, seed).

val_df = df.drop_duplicates(['proc', 'seed'])[['proc', 'seed', 'r_val', 'h_val']].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
fig.suptitle('NLL de Validação por Semente', fontsize=13, fontweight='bold')

for ax, (pname, proc) in zip(axes, PROCESSES):
    sub = val_df[val_df['proc'] == pname].reset_index(drop=True)
    x   = np.arange(len(sub))

    ax.scatter(x - 0.12, sub['r_val'], color=COR_ROTHP, s=80, zorder=3,
               label='RoTHP', edgecolors='white', linewidths=0.5)
    ax.scatter(x + 0.12, sub['h_val'], color=COR_HOTHP, s=80, zorder=3,
               label='HoTHP', edgecolors='white', linewidths=0.5)

    ax.set_title(proc['label'], fontsize=12)
    ax.set_xlabel('Índice da semente')
    ax.set_ylabel('NLL de validação')
    ax.set_xticks(x)
    ax.set_xticklabels([f's{i}' for i in range(len(sub))], fontsize=8)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico de barras pareado: NLL absoluta por dataset
# Linha superior: NLL com máscara (só eventos OOD — nossa métrica)
# Linha inferior: NLL sem máscara (sequência inteira — forma comum na literatura)
# Comparar as duas linhas permite verificar se excluir os eventos in-dist faz diferença.

categories = ['1×\n(in-dist)', '2×\n(OOD)', '5×\n(OOD)', '10×\n(OOD)']
bar_w = 0.35
x = np.arange(len(categories))

# Dois conjuntos de plots: [0] = mascarado (só OOD), [1] = sequência inteira
variants = [
    ('r_ood',      'h_ood',      'Só eventos OOD (mascarado)'),
    ('r_ood_full', 'h_ood_full', 'Sequência inteira (sem máscara)'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for row, (r_col, h_col, variant_label) in enumerate(variants):
    for col, (pname, proc) in enumerate(PROCESSES):
        ax  = axes[row, col]
        sub = df[df['proc'] == pname]

        # 1×: usar r_short/h_short (uma linha por seed)
        inDist = sub.drop_duplicates('seed')[['r_short', 'h_short']]
        r_means = [inDist['r_short'].mean()]
        h_means = [inDist['h_short'].mean()]
        r_stds  = [inDist['r_short'].std()]
        h_stds  = [inDist['h_short'].std()]

        # 2×, 5×, 10×
        for f in EXTRAP_FACTORS:
            grp = sub[sub['factor'] == f]
            r_means.append(grp[r_col].mean())
            h_means.append(grp[h_col].mean())
            r_stds.append(grp[r_col].std())
            h_stds.append(grp[h_col].std())

        r_means = np.array(r_means)
        h_means = np.array(h_means)
        r_stds  = np.array(r_stds)
        h_stds  = np.array(h_stds)

        ax.bar(x - bar_w/2, r_means, bar_w, yerr=r_stds,
               color=COR_ROTHP, label='RoTHP', capsize=4, alpha=0.85)
        ax.bar(x + bar_w/2, h_means, bar_w, yerr=h_stds,
               color=COR_HOTHP, label='HoTHP', capsize=4, alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(categories, fontsize=10)
        ax.set_xlabel('Dataset', fontsize=10)
        ax.set_ylabel('NLL (por evento)', fontsize=10)
        ax.set_title(f'{proc["label"]}\n({variant_label})', fontsize=11, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

fig.suptitle(
    f'HoTHP vs RoTHP — NLL Absoluta por Dataset\n'
    f'n={N_SEEDS} sementes  |  barras = média ± dp',
    fontsize=12
)
plt.tight_layout()
plt.show()
